# **Processing PubMedQA dataset**

## **PubMedQA Label Distribution Analysis**

This script loads the dataset and counts every unique value found in the final_decision field.

In [ ]:
import json
import os
from collections import Counter

INPUT_FILE = "../../../Data/PubmedQA/ori_pqal.json"

def analyze_pubmedqa_labels():
    print(f" Analyzing Label Distribution in: {INPUT_FILE}...\n")

    if not os.path.exists(INPUT_FILE):
        print(f" Error: File '{INPUT_FILE}' not found.")
        return

    label_counts = Counter()
    total_entries = 0

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)

            for study_id, content in data.items():
                total_entries += 1
                
                label = content.get("final_decision", "MISSING").lower()
                
                label_counts[label] += 1

        print(f"   Total Entries: {total_entries}")
        print("-" * 40)
        print(f"   Unique Labels Found:")
        
        for label, count in label_counts.most_common():
            percentage = (count / total_entries) * 100
            print(f"   - '{label}': {count:<5} ({percentage:.1f}%)")
            
        print("-" * 40)
        
    except Exception as e:
        print(f" An unexpected error occurred: {e}")

analyze_pubmedqa_labels()

 Analyzing Label Distribution in: ../../../Data/PubmedQA/ori_pqal.json...

   Total Entries: 1000
----------------------------------------
   Unique Labels Found:
   - 'yes': 552   (55.2%)
   - 'no': 338   (33.8%)
   - 'maybe': 110   (11.0%)
----------------------------------------


## **PubMedQA Test Set Formatter**

Formats the dataset to use in evaluation

In [ ]:
import json
import os

INPUT_FILE = "../../../Data/PubmedQA/ori_pqal.json" 

OUTPUT_PATH = "pubmedqa_test.jsonl"

SYSTEM_INSTRUCTION = "You are a clinical decision support assistant. Determine whether the following statement is correct. Answer ONLY with 'yes', 'no', or 'maybe'."

def format_pubmedqa_test():
    print(f"Starting PubMedQA Test Set Conversion (3-Class Mode)...")
    print(f"   Input: {INPUT_FILE}")

    if not os.path.exists(INPUT_FILE):
        print(f" Error: Input file '{INPUT_FILE}' not found.")
        return

    converted_data = []
    stats = {"yes": 0, "no": 0, "maybe": 0}

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f) 

            for study_id, content in data.items():
                
                question = content.get("QUESTION", "").strip()
                
                # Context is a list of strings, it is necessary to join them
                context_list = content.get("CONTEXTS", [])
                context_text = " ".join(context_list).strip()
                
                final_decision = content.get("final_decision", "").lower()

                if final_decision in stats:
                    stats[final_decision] += 1
               
                formatted_input = (
                    f"###Question: {question}\n"
                    f"###Context: {context_text}"
                )

                entry = {
                    "instruction": SYSTEM_INSTRUCTION,
                    "input": formatted_input,
                    "output": final_decision 
                }
                
                converted_data.append(entry)

        print(f" Saving {len(converted_data)} entries to '{OUTPUT_PATH}'...")
        
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
            for item in converted_data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

        print("-" * 40)
        print(f" Conversion complete")
        print("-" * 40)
        print(f"   Total Questions: {len(converted_data)}")
        print(f"   Answers:")
        print(f"     - Yes:   {stats['yes']}")
        print(f"     - No:    {stats['no']}")
        print(f"     - Maybe: {stats['maybe']}")
        print("-" * 40)

    except Exception as e:
        print(f" Error: {e}")

format_pubmedqa_test()

Starting PubMedQA Test Set Conversion (3-Class Mode)...
   Input: ../../../Data/PubmedQA/ori_pqal.json
 Saving 1000 entries to 'pubmedqa_test.jsonl'...
----------------------------------------
 Conversion complete
----------------------------------------
   Total Questions: 1000
   Answers:
     - Yes:   552
     - No:    338
     - Maybe: 110
----------------------------------------


## **Few-shot prompting**
Choosed manually with 5 entries and the entries were choosen to have "yes" and "no". In the pqa-a dataset doesnt exist "maybe" answers. 